## PREPARING DATA

In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import normalize

In [ ]:
from matplotlib.ticker import MultipleLocator


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent.parent
    print(script_dir)
    data_folder = script_dir /'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG_Original = timeSeriesData_BIGraw.copy()
userInputData_Original = userInputDataRaw.copy()

In [ ]:
userInputData_Original

In [ ]:
userInputData_Original["room"].unique()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space d
room_mask = userInputData_Original["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'])#,'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'])
mask = room_mask # & (userInputData_Original["are-doors-opened"]!="on")
userInputData_Original = userInputData_Original.loc[mask]
timeSeriesData_BIG_Original = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.index)]

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
column_to_transform = "x-y axis"
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                    for id in range(3)]

userInputData_Original.loc[:,sensors_position_columns] = userInputData_Original.loc[:,sensors_position_columns].applymap(lambda lst: 
                                                                                                                         [round(x,2) for x in lst if x is not None]
                                                                                                                         if lst is not None
                                                                                                                         else lst
                                                                                                                        )
userInputData_Original.loc[:,sensors_euclidian_distance_columns] = userInputData_Original.loc[:,sensors_euclidian_distance_columns].round(2)


In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
for column_to_transform in sensors_position_columns:
    userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(
                                                                                                     lambda x: tuple(x)
                                                                                                     if x is not None      
                                                                                                     else x     
                                                                                                    )

In [ ]:
userInputData_Original.loc[userInputData_Original['room']=='Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'].head(20)

In [ ]:
userInputData_Original['position of Id=2:BME680:breathVocEquivalent x-y axis'].unique()

In [ ]:
userInputData_Original.columns

In [ ]:

euclidian_distance_columns = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                               for id in range(3) ]

mask = userInputData_Original["are-doors-opened"] == "on"
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] == on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")
mask = userInputData_Original["are-doors-opened"] != "on"    
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] != on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]=="on"].shape

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]!="on"].shape

In [ ]:
max_before= -30
max_after = 299

column_to_check = "time taken before insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] >max_before
print(f"{column_to_check} < {max_before}:\n {userInputData_Original.loc[mask,column_to_check]}")

column_to_check = "time taken after insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] < max_after
print(f"{column_to_check} < {max_after}:\n {userInputData_Original.loc[mask,column_to_check]}")
print(f"userInputData_Original rows {userInputData_Original.shape[0]}")

In [ ]:
userInputData_Original.columns

## CREATE NEW COLUMNS AT TIME SERIES

In [ ]:
#savgol_filter window 60 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed"
column_to_take = "VOC original"

timeSeriesData_BIG_Original[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG_Original.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG_Original[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG_Original[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG_Original.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG_Original.loc[mask,column_to_take],60,2)

sensors = timeSeriesData_BIG_Original["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("VOC smoothed savgol_filter 60 2")
 #   plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor) 

create MIN MAX SCALER

from sklearn.preprocessing import MinMaxScaler
group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed MinMaxScaler"
column_to_take = "VOC smoothed"

timeSeriesData_BIG_Original[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG_Original.groupby(group_by_columns).groups:
    scaler = MinMaxScaler()
    mask = (timeSeriesData_BIG_Original[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG_Original[group_by_columns[1]] == key_sensor[1])

    timeSeriesData_BIG_Original.loc[mask,column_to_create] = scaler.fit_transform(timeSeriesData_BIG_Original.loc[mask,column_to_take].to_numpy().reshape(-1,1))


create Gradient of smoothed data

group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed Gradient"
column_to_take = "VOC smoothed"

timeSeriesData_BIG_Original[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG_Original.groupby(group_by_columns).groups:
    scaler = MinMaxScaler()
    mask = (timeSeriesData_BIG_Original[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG_Original[group_by_columns[1]] == key_sensor[1])

    timeSeriesData_BIG_Original.loc[mask,column_to_create] = np.gradient(timeSeriesData_BIG_Original.loc[mask,column_to_take].to_numpy())


group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed MinMaxScaler Gradient"
column_to_take = "VOC smoothed MinMaxScaler"

timeSeriesData_BIG_Original[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG_Original.groupby(group_by_columns).groups:
    scaler = MinMaxScaler()
    mask = (timeSeriesData_BIG_Original[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG_Original[group_by_columns[1]] == key_sensor[1])

    timeSeriesData_BIG_Original.loc[mask,column_to_create] = np.gradient(timeSeriesData_BIG_Original.loc[mask,column_to_take].to_numpy())


group_by_columns = ["keys","sensors"]
column_to_create = "bigger than previous Gradients"
column_to_take = "VOC smoothed Gradient"

timeSeriesData_BIG_Original[column_to_create] = 0
for key_sensor in timeSeriesData_BIG_Original.groupby(group_by_columns).groups:
    scaler = MinMaxScaler()
    mask = (timeSeriesData_BIG_Original[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG_Original[group_by_columns[1]] == key_sensor[1])
    for cur_second in range(15,timeSeriesData_BIG_Original.loc[mask,"seconds"].max()):
        
        curr_gradient_mask = mask & (timeSeriesData_BIG_Original["seconds"]==cur_second)
        
        mean_gradient_mask = mask & ((timeSeriesData_BIG_Original["seconds"]<cur_second) & (timeSeriesData_BIG_Original["seconds"]>=cur_second - 45))
        mean_gradient = timeSeriesData_BIG_Original.loc[mean_gradient_mask,column_to_take].mean()
        if (mean_gradient< timeSeriesData_BIG_Original.loc[curr_gradient_mask,column_to_take]).all():
            
            timeSeriesData_BIG_Original.loc[curr_gradient_mask,column_to_create] = 1


In [ ]:
timeSeriesData_BIG_Original

timeSeriesData_BIG_Original["VOC smoothed MinMaxScaler Gradient"].max()

## PRINT FUNCTIONS TO USE

In [ ]:
def plot_position(userInputData,sample_row_of_the_group):
    plt.figure(figsize=(6, 6))  
    position_of_sensors = userInputData.iloc[-1]
    all_positions = userInputData.loc[:, ["x axis", "y axis"]]
    # Extra points
    extra_positions = np.array([
        [position_of_sensors["position of Id=0:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=0:BME680:breathVocEquivalent-y axis"]],
    
        [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
        [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
    ])
    extra_ids = ["id0","id1", "id2"]
    
    
    room_length = 4.0
    room_width = 3.25
    
    
    # Create scatterplot of the sources of all the particular setup
    #sns.scatterplot(x=positions[:,0], y=positions[:,1])
    sns.scatterplot(data=all_positions, x="x axis", y="y axis", color='blue', s=100, label='User Input Data')
    
    
    
    # Add the positions of sensors
    sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)
    x_sensor_highlight = sample_row_of_the_group["x axis"]
    y_sensor_highlight = sample_row_of_the_group["y axis"]
    # Plot a hollow circle around it
    plt.scatter(x_sensor_highlight, y_sensor_highlight, s=500, facecolors='none', edgecolors='green', linewidths=2, label='Highlighted point')  
    # Draw lines and annotate distances
    distances_from_sensors = (
        sample_row_of_the_group["Euclidian distance to Id=0:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=1:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=2:BME680:breathVocEquivalent"]
    )
    
    for i, (x, y) in enumerate(extra_positions):
        plt.plot([x_sensor_highlight, x], [y_sensor_highlight, y], color='red', linewidth=0.7, alpha=0.7)
        
        if distances_from_sensors is not None:
            # Midpoint of the line for annotation
            mid_x = (x_sensor_highlight + x) / 2
            mid_y = (y_sensor_highlight + y) / 2
            plt.text(mid_x, mid_y, f"{distances_from_sensors[i]:.2f}", color='red', fontsize=8, ha='center', va='center',
                         bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, pad=1))
    # Annotate extra points with their IDs
    for i, (x, y) in enumerate(extra_positions):
        plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')
    
    
    # Set axis limits
    plt.xlim(-room_width, 0)
    plt.ylim(0, room_length)
    
    # Add grid
    plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
    # Smaller legend
    plt.legend(fontsize=8, markerscale=0.8, labelspacing=0.4)

    plt.show()

In [ ]:
def printDataBasedOnDate(column_to_print,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping,minmax=False):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        timeSeriesData_BIG_copy = timeSeriesData_BIG.copy() 
        if ("position"  in type_of_other_grouping):
            sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
            
            plot_position(userInputData,sample_row_of_the_group)       
        print(f"group_name {group_name}")
        print(f"indexes_of_the_group {indexes_of_the_group}")
        data = timeSeriesData_BIG_copy.loc[timeSeriesData_BIG_copy["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        
        if minmax is True:
                g.set(ylim=(0, 0.6))
        # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
            
 
        
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
           
                #value to show the time that source is inserted
          
            userInputDataRow = userInputData.loc[key_value,:]
        #    x_position = f"side-right-wall {userInputDataRow['side-right-wall']},side-left-wall {userInputDataRow['side-left-wall']} \n"
        #    y_position = f"front-wall {userInputDataRow['front-wall']},back-wall {userInputDataRow['back-wall']} \n"
            
            euclidian_distances = (
                                  f"distance from Id0 sensor {userInputDataRow['Euclidian distance to Id=0:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id1 sensor {userInputDataRow['Euclidian distance to Id=1:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id2 sensor {userInputDataRow['Euclidian distance to Id=2:BME680:breathVocEquivalent']}\n",
            )
            subtitle=  f"At experiment with key {key_value} \n datetime:{userInputDataRow['actual timestamp StartingExperiment']}\n" 
            subtitle= subtitle + f"experimentState:{userInputDataRow['experimentState']}\n"
            subtitle= subtitle + f"x-axis: {userInputDataRow['x axis']} , y-axis: {userInputDataRow['y axis']}\n"
            subtitle= subtitle + f"and have door {userInputDataRow['are-doors-opened']}\n"
            if ("distance"  in type_of_other_grouping):
               subtitle = "".join(subtitle) + "\n"+"".join(euclidian_distances)  
            ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
            top=0.75,
            wspace=0.2,
            hspace=0.8   # ← increase this
        )


        plt.show()   
             

In [ ]:
def plot_all_positions(userInputData):
    room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
        plot_position(userInputData,sample_row_of_the_group)      
        
plot_all_positions(userInputData_Original)

In [ ]:
def plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor,minmax=False):
    sensors = timeSeriesData_BIG["sensors"].unique()
    euclidian_distances_column = f"Euclidian distance to {sensor}" 
    group_by_list = ["room","experimentState",euclidian_distances_column]
    room_other_grouping = userInputData.groupby(group_by_list).groups
    type_of_other_grouping = ["experimentState","position","distance"]
    
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    printDataBasedOnDate(column_to_take,userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping,minmax=minmax)

## PRINT ALL DATA

### PLANT DATA ALL TOGETHER

In [ ]:
room_other_grouping = userInputData_Original.groupby(["date of experiment"]).groups
type_of_other_grouping = ["date of experiment"]
column_to_take = "VOC smoothed"
userInputData_Original_temp = userInputData_Original.loc[(userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')] # & (userInputData_Original["are-doors-opened"]!="on")] 
printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

In [ ]:
room_other_grouping = userInputData_Original.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
column_to_take = "VOC smoothed"
userInputData_Original_temp = userInputData_Original.loc[(userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')] #  & (userInputData_Original["are-doors-opened"]!="on")] 
printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

room_other_grouping = userInputData_Original.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
column_to_take = "VOC smoothed Gradient"
userInputData_Original_temp = userInputData_Original.loc[(userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')] #  & (userInputData_Original["are-doors-opened"]!="on")] 
printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

room_other_grouping = userInputData_Original.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
column_to_take = "VOC smoothed MinMaxScaler"
userInputData_Original_temp = userInputData_Original.loc[(userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')] #  & (userInputData_Original["are-doors-opened"]!="on")] 
printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

room_other_grouping = userInputData_Original.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
column_to_take = "VOC smoothed MinMaxScaler Gradient"
userInputData_Original_temp = userInputData_Original.loc[(userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')] #  & (userInputData_Original["are-doors-opened"]!="on")] 
printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

### PLANT DATA PER SENSOR

In [ ]:
sensors = timeSeriesData_BIG_Original["sensors"].unique()
sensors

#### Id=0:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC smoothed"
sensor='Id=0:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor)

column_to_take = "VOC smoothed Gradient"
sensor='Id=0:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor,minmax = True)

column_to_take = "VOC smoothed MinMaxScaler"
sensor='Id=0:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor)

column_to_take = "VOC smoothed MinMaxScaler Gradient"
sensor='Id=0:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor,minmax=True)

#### Id=1:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC smoothed"
sensor='Id=1:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor)

#### Id=2:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC smoothed"
sensor='Id=2:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor)

# PRINT CLOSED DOOR 

### PLANT DATA ALL TOGETHER

#### grabClosestDataPerSensor

In [ ]:
def grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=True):
    #grab only for most recent roomsetup
    mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' & (userInputData_Original["are-doors-opened"]!="on")
    userInputData = userInputData_Original.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData.index)].copy()
    
    #grab the data closest to each sensor
    
    #start from just an empty boolean expression and add the conditions
    mask = pd.Series(False, index=userInputData.index)
    columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
    for column_to_grab in columns_to_grab:
        
        smallest_distances = np.sort(userInputData.loc[userInputData[column_to_grab] != None,column_to_grab].unique())
        print(smallest_distances[:number_of_distances_to_grab])
        if grab_closest:
            mask = mask | (userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
        else:
            mask = mask | (~userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
            
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
    return userInputData,timeSeriesData_BIG

#### CHECK DATA

In [ ]:
number_of_distances_to_grab = 2
userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=True)
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:

column_to_take = "VOC smoothed"
mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')  & (userInputData_Original["are-doors-opened"]!="on")
userInputData_Original_temp = userInputData_Original.loc[mask] 

room_other_grouping = userInputData_Original_temp.groupby(["date of experiment"]).groups
type_of_other_grouping = ["date of experiment"]
printDataBasedOnDate(column_to_take,userInputData_Original_temp,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:

column_to_take = "VOC smoothed"
mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')  & (userInputData_Original["are-doors-opened"]!="on")
userInputData_Original_temp = userInputData_Original.loc[mask] 

room_other_grouping = userInputData_Original_temp.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate(column_to_take,userInputData_Original_temp,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

### PLANT DATA PER SENSOR

#### Id=0:BME680:breathVocEquivalent

In [ ]:

sensor='Id=0:BME680:breathVocEquivalent'
column_to_take = "VOC smoothed"
mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')  & (userInputData_Original["are-doors-opened"]!="on")
userInputData_Original_temp = userInputData_Original.loc[mask] 
plantDataGroupedByEuclidianDistance(userInputData_Original_temp,timeSeriesData_BIG_Original,column_to_take,sensor)

#### Id=1:BME680:breathVocEquivalent

In [ ]:
sensor='Id=1:BME680:breathVocEquivalent'
column_to_take = "VOC smoothed"
mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')  & (userInputData_Original["are-doors-opened"]!="on")
userInputData_Original_temp = userInputData_Original.loc[mask] 

plantDataGroupedByEuclidianDistance(userInputData_Original_temp,timeSeriesData_BIG_Original,column_to_take,sensor)

#### Id=2:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC smoothed"
sensor='Id=2:BME680:breathVocEquivalent'
mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0')  & (userInputData_Original["are-doors-opened"]!="on")
userInputData_Original_temp = userInputData_Original.loc[mask] 

plantDataGroupedByEuclidianDistance(userInputData_Original_temp,timeSeriesData_BIG_Original,column_to_take,sensor)

# SEARCH SOME STUFF (DON'T RUN IT)

## GRAB THE DATA

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
userInputData_Original["room"].unique()

In [ ]:
# we will keep the closest places 
places_to_keep = []
columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
for column_to_grab in columns_to_grab:
    print(userInputData_Original.loc[:,column_to_grab].min())
    
    

In [ ]:
def grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=True):
    #grab only for most recent roomsetup
    mask = (userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0') & (userInputData_Original["are-doors-opened"] != "on")
    userInputData = userInputData_Original.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData.index)].copy()
    
    #grab the data closest to each sensor
    
    #start from just an empty boolean expression and add the conditions
    mask = pd.Series(False, index=userInputData.index)
    columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
    for column_to_grab in columns_to_grab:
        
        smallest_distances = np.sort(userInputData.loc[userInputData[column_to_grab] != None,column_to_grab].unique())
        print(smallest_distances[:number_of_distances_to_grab])
        if grab_closest:
            mask = mask | (userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
        else:
            mask = mask | (~userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
            
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
    return userInputData,timeSeriesData_BIG
number_of_distances_to_grab = 2    
userInputData,timeSeriesData_BIG =    grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=True) 

In [ ]:
userInputData

In [ ]:
userInputData.shape

In [ ]:
timeSeriesData_BIG

## PLANT THE RAW DATA 

### PLANT ALL DATA

In [ ]:
room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC original",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

#### Id=0:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC original"
sensor='Id=0:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)

#### Id=1:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC original"
sensor='Id=1:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)

#### Id=2:BME680:breathVocEquivalent

In [ ]:
column_to_take = "VOC original"
sensor='Id=2:BME680:breathVocEquivalent'
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor)

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = "VOC original"
for sensor in sensors:
    print(column_to_take)
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = "VOC rolling average"
for sensor in sensors:
    print(column_to_take)
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)

## SMOOTH THE DATA

GRAB THE ID=0 FIRST
kwargs = {}
autocorr

In [ ]:
 
def findFirstLagValueUnderThershold(row,thereshold = 0.5):
    thereshold = 0.5
    first_occurance_lag = None
    for lag in row.index:
        if first_occurance_lag is None and row[lag] <= thereshold:
            first_occurance_lag = float(lag.split()[-1])
            break
    return first_occurance_lag   

    
def getAuto_Matrix_And_First_occurance(userInputData,timeSeriesData_BIG,sensor):
    #grab the first one
    lag_size = 60 + 1
    
    column_to_take = "VOC original"
    #userInput_mask = userInputData[column_name] == userInputData[column_name].min()
    userInput_Test =userInputData
    timeSeriesData_BIG_mask = (timeSeriesData_BIG["sensors"]==sensor) # & (timeSeriesData_BIG["keys"].isin(userInput_Test.index))
    timeSeriesData_BIG_Test = timeSeriesData_BIG.loc[timeSeriesData_BIG_mask]
    columns_to_lag = ['lag- '+str(i) for i in np.arange(lag_size)]
    lag_df = pd.DataFrame(data=None,index=userInput_Test.index,columns=columns_to_lag)
    print(timeSeriesData_BIG_Test["sensors"].unique())
    for index in userInput_Test.index:
        for lag in np.arange(lag_size):
            lag_column = 'lag- '+str(lag)
            mask = timeSeriesData_BIG_Test["keys"] == index
            lag_df.loc[index,lag_column] = timeSeriesData_BIG_Test.loc[mask,column_to_take].autocorr(lag=lag)
    first_pos = pd.Series(data=None,index=lag_df.index)
    first_pos = lag_df.apply(findFirstLagValueUnderThershold,axis=1)
        
    return lag_df,first_pos


In [ ]:
sensor= 'Id=0:BME680:breathVocEquivalent'
lag_df,first_pos  =  getAuto_Matrix_And_First_occurance(userInputData,timeSeriesData_BIG,sensor)
print(first_pos.mean())
first_pos

In [ ]:
sensor= 'Id=1:BME680:breathVocEquivalent'

lag_df,first_pos  =  getAuto_Matrix_And_First_occurance(userInputData,timeSeriesData_BIG,sensor)
print(first_pos.mean())

first_pos

In [ ]:
sensor= 'Id=2:BME680:breathVocEquivalent'

lag_df,first_pos  =  getAuto_Matrix_And_First_occurance(userInputData,timeSeriesData_BIG,sensor)
print(first_pos.mean())

first_pos

In [ ]:
from scipy.signal import savgol_filter
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "30"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],30,2)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter 2")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = "30"
for sensor in sensors:
    print("savgol_filter 2")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "602"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],60,2)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter 60 2")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "60"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],60,3)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter 3")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "60"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],60,4)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter 4444")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "60"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],90,2)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter 5555555")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "120"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],120,2)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter aaaaaaaaaaaa")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "120"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],90,3)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter aaaaaaaa111111111aaaa")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "120"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],30,3)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("savgol_filter 30 33333333")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#savgol_filter window 28 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "120"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],45,2)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("45")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#ry exponentialt
group_by_columns = ["keys","sensors"]
column_to_create = "aba"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = timeSeriesData_BIG.loc[mask,column_to_take].ewm(span=15).mean()

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("exponential 60")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

60/2 h 60/3 fainetai to kalitero apotelsma

In [ ]:
#ry exponentialt
group_by_columns = ["keys","sensors"]
column_to_create = "aba"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = timeSeriesData_BIG.loc[mask,column_to_take].ewm(span=30).mean()

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("exponential 30")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#ry exponentialt
group_by_columns = ["keys","sensors"]
column_to_create = "aba"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = timeSeriesData_BIG.loc[mask,column_to_take].ewm(span=60).mean()

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("exponential 60")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

60/2 or 60/3 best doesn't matter,perhaps 60/2 more secure

## END SMOOTHING DATA

## CHECK MIN MAX SCALER AND NORMALIZER AT SMOOTHED DATA

In [ ]:
from scipy.signal import savgol_filter


In [ ]:
#savgol_filter window 60 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed"
column_to_take = "VOC original"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    

    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = savgol_filter(timeSeriesData_BIG.loc[mask,column_to_take],60,2)

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("VOC smoothed savgol_filter 60 2")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take,sensor)   

In [ ]:
#MIN MAX SCALER


In [ ]:
timeSeriesData_BIG = timeSeriesData_BIG.rename(columns={"seconds": "seconds_col"})

In [ ]:
#savgol_filter window 60 pol 2 
group_by_columns = ["seconds","sensors"]
column_to_create = "VOC smoothed-Min Max"
column_to_take = "VOC smoothed"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    
    scaler = MinMaxScaler()
    mask = (timeSeriesData_BIG.index == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = scaler.fit_transform(timeSeriesData_BIG.loc[mask,column_to_take].to_numpy().reshape(-1, 1))

sensors = timeSeriesData_BIG["sensors"].unique()
column_to_take = column_to_create
for sensor in sensors:
    print("VOC smoothed-Min Max")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_create,sensor)   

In [ ]:
room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC smoothed-Min Max",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
# NORMALIZER


In [ ]:
#savgol_filter window 60 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed-Normalized"
column_to_take = "VOC smoothed"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    
    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = normalize(timeSeriesData_BIG.loc[mask,column_to_take].to_numpy().reshape(-1, 1),axis=0)

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    print("VOC smoothed-Normalized")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_create,sensor)   

In [ ]:
room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
printDataBasedOnDate("VOC smoothed-Normalized",userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)

In [ ]:
#savgol_filter window 60 pol 2 
group_by_columns = ["keys","sensors"]
column_to_create = "VOC smoothed-Normalized"
column_to_take = "VOC smoothed"

timeSeriesData_BIG[column_to_create] = np.nan
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    
    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    timeSeriesData_BIG.loc[mask,column_to_create] = normalize(timeSeriesData_BIG.loc[mask,column_to_take].to_numpy().reshape(-1, 1),axis=0)

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    print("VOC smoothed-Normalized")
    plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_create,sensor)   

## GET VARIANCE AND MEAN

In [ ]:
group_by_columns = ["keys","sensors"]
column_to_take = "VOC smoothed-Normalized"
sensors = timeSeriesData_BIG["sensors"].unique()

mean_sensors = ['mean-' + sensor for sensor in sensors]
variance_sensors =['variance-' + 'sensor' for sensor in sensors]
columns = [*mean_sensors,*variance_sensors]
mean_and_variance = pd.DataFrame(data=np.nan,columns=columns,index=userInputData.index)
for key_sensor in timeSeriesData_BIG.groupby(group_by_columns).groups:
    
    mask = (timeSeriesData_BIG[group_by_columns[0]] == key_sensor[0]) & (timeSeriesData_BIG[group_by_columns[1]] == key_sensor[1])
    mean_and_variance.loc[key_sensor[0],'mean-' +key_sensor[1]] = timeSeriesData_BIG.loc[mask,column_to_take].mean()
    mean_and_variance.loc[key_sensor[0],'variance-' +key_sensor[1]] = timeSeriesData_BIG.loc[mask,column_to_take].var()
mean_and_variance

## GET DTW SIMILARITY

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
def heatmapDTWMatrix(sensor,target_variable,dtw_array):
    # Correlation matrix
   # print("rows_of_data shape:", rows_of_data.shape)
   # corr_matrix = np.corrcoef(rows_of_data)
    # Heatmap
    print(corr_matrix)
    plt.figure(figsize=(13, 8))  # square & larger

    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        annot_kws={"size": 10},   # 👈 key fix
        cbar_kws={"label": "Correlation"}
    )
    
    plt.title(f"Correlation Matrix Heatmap for sensor with id={sensor},at euclidian distance {target_variable}")
    plt.show()

In [ ]:
from tslearn.metrics import dtw, dtw_path
from tslearn.utils import to_time_series
sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances = ["Euclidian distance to "+str(sensor) for sensor in sensors]
column_to_take = "VOC smoothed"
group_by_columns = ["keys","sensors"]

for sensor,euclidian_distance in zip(sensors,euclidian_distances):
    print(f"for {sensor}")

    for distance,relevant_indexes in userInputData.groupby(column_to_grab).groups.items():
        print(f"for {distance}")
        df = pd.DataFrame()
        for df_index in relevant_indexes:
            for df_column in relevant_indexes:
                timeseries_index = to_time_series(timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"] == df_index,column_to_take].to_numpy())
                #print(timeseries_index.shape)
                timeseries_column = to_time_series(timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"] == df_column,column_to_take].to_numpy())
                #print(timeseries_column.shape)
                
                dtw_score = dtw(timeseries_index, timeseries_column)
                df.loc[df_index,df_column] =  dtw_score 
        heatmapDTWMatrix(sensor,target_variable,df)        

## END SEARCH FED DATA

# LIBRARIES AND FUNCTIONS DECLARACTIONS

## LIBRARIES

In [ ]:
from sklearn.manifold import TSNE# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score

import math
from itertools import combinations
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import normalize

from scipy.optimize import least_squares
from tslearn.metrics import dtw, dtw_path
from tslearn.utils import to_time_series
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.manifold import TSNE

from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

## MODELS

In [ ]:

# 1. Linear Regression (no hyperparameters to tune, just for completeness)
lr = LinearRegression()
lr_params = {}  # No parameters for simple LinearRegression

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# 3. Lasso Regression
lasso = Lasso()
lasso_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
}

# 4. ElasticNet
elastic = ElasticNet()
elastic_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# 5. Support Vector Regression
svr = SVR()
svr_params = {
    'kernel': ['linear','rbf'],
    'C': [0.1, 1, 10, 100],
    'epsilon': [0.001, 0.01, 0.05, 0.1, 0.2],
    'gamma': ['auto',1e-3, 1e-2, 1e-1, 1,'scale'],
   # 'shrinking': [True, False],
 #   'tol': [1e-4, 1e-3]
}


# 7. Random Forest Regression
rf = RandomForestRegressor()
rf_params = {
    'random_state':[42],
    'n_estimators': [10, 50,200,400],
    'max_depth': [None, 5,30,50],
    'min_samples_split': [1,2, 5],
    'min_samples_leaf': [1, 2, 10]
}

# 8. Gradient Boosting Regression
gbr = GradientBoostingRegressor()
gbr_params = {
    'random_state':[42],
    'n_estimators': [10, 50,200,400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [None, 5,30,50],
    'min_samples_split': [1,2, 5],
    'min_samples_leaf': [1, 2, 10]
}

models = {
    'LinearRegression': (lr, lr_params),
    'Ridge': (ridge, ridge_params),
    'Lasso': (lasso, lasso_params),
    'ElasticNet': (elastic, elastic_params),
    'SVR': (svr, svr_params),
#    'RandomForest': (rf, rf_params),
#    'GradientBoosting': (gbr, gbr_params)
}



#### CLASSIFICATION MODELS

In [ ]:

# 1. Logistic Regression
logreg = LogisticRegression(max_iter=5000)
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],          # 'l1' requires solver='liblinear' or 'saga'
    "solver": ["lbfgs", "saga"]
}

# 2. Support Vector Classifier
svc = SVC()
svc_params = {
    "kernel": ["rbf"],
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto"]
}

# 3. K-Nearest Neighbors
knn = KNeighborsClassifier()
knn_params = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

# 4. Decision Tree Classifier
dt = DecisionTreeClassifier()
dt_params = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 5. Random Forest Classifier
rf = RandomForestClassifier(random_state=42)
rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 6. Gradient Boosting Classifier
gbc = GradientBoostingClassifier()
gbc_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# 7. Neural Network (MLPClassifier)
mlp = MLPClassifier(max_iter=3000)
mlp_params = {
    "hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01]
}


# Combine them into your expected dictionary:
classification_models = {
    "LogisticRegression": (logreg, logreg_params),
    "SVC": (svc, svc_params),
    "KNN": (knn, knn_params),
  #  "DecisionTree": (dt, dt_params),
  #  "RandomForest": (rf, rf_params),
 #   "GradientBoosting": (gbc, gbc_params),
  #  "MLPClassifier": (mlp, mlp_params),
}



## FUNCTIONS

### REGRESSION FUNCTIONS

#### get_16_train_positions

In [ ]:
def get_16_train_positions(userInputData_Original:pd.DataFrame):
    column_to_check = "x-y axis"
    mask = (userInputData_Original["are-doors-opened"] != "on") & (userInputData_Original["room"] =='Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' ) & (userInputData_Original[column_to_check] != (None,None))
    training_positions = userInputData_Original.loc[mask,column_to_check].unique()
    return training_positions


#### get_Data_To_Be_Used

In [ ]:
def get_Data_To_Be_Used(userInputData_Original:pd.DataFrame,timeSeriesData_BIG_Original:pd.DataFrame,
                        keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)->(pd.DataFrame,pd.DataFrame):
   # print(f"drop_other_positions:{drop_other_positions}")
    userInputData = userInputData_Original.copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.copy()
    # clear the data that doesn't fit to the 330 columns we want
    max_before_int= -30
    max_after_int = 299
    
    mask_before = (userInputData["time taken before insertion (seconds)-capped"] == max_before_int)
    max_after = (userInputData["time taken after insertion (seconds)-capped"] == max_after_int)
    mask = mask_before & max_after
 #   print(f"data which doesn't fit our criteria of size:{userInputData.loc[~mask,:].index}")
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    if keepOpenDoorData:
        mask = (userInputData["are-doors-opened"] == "on") | (userInputData["are-doors-opened"] != "on")

    else:
        mask =  userInputData["are-doors-opened"] != "on"

    if anyOtherMaskToBeUsed is not None:
        mask = mask & anyOtherMaskToBeUsed
    userInputData = userInputData.loc[mask].copy()
    #grab all the data that are contained in those experiments
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
        
   # print(f"userInputData['x-y axis']{userInputData['x-y axis']}")
    if drop_other_positions is True & (not closeDoorTraining_openDoorTest):
        #from the experiments with door open, drop the positions which doesn't fit to the 16 source position we are going to check the model
        #also drop the experiments with the None values
        axis_list = get_16_train_positions(userInputData_Original)
        axis_mask = userInputData["x-y axis"].isin(axis_list)
        
        mask = axis_mask 
        userInputData = userInputData.loc[mask]
        #grab all the data that are contained in those experiments
        timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    elif drop_other_positions is False:
        axis_mask = userInputData["x-y axis"].isin([(None,None)])
        mask = ~axis_mask 
        userInputData = userInputData.loc[mask]
        #grab all the data that are contained in those experiments
        timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    return userInputData,timeSeriesData_BIG
        

#### build_X_keys_And_Y_target_Arrays

In [ ]:
def build_X_keys_And_Y_target_Arrays(userInputData,closeDoorTraining_openDoorTest = False)->(np.array,np.array):
  
    if closeDoorTraining_openDoorTest :
         pass

    else:
        keys = np.array(userInputData.index)
        keys_numpy = keys.reshape(-1,1)
        columns_to_pass = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                                   for id in range(3) ]
        column_size = len(columns_to_pass)
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_euclidian_distance_numpy = target_variables.reshape(-1,column_size)
     
        # print("aaaa")
        # print(target_variables_euclidian_distance_numpy)
        # print(np.unique(target_variables_euclidian_distance_numpy, axis=0))
        columns_to_pass = ["x-y axis"]
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_positions_numpy = np.stack(target_variables[:,0])  
        # print("bbbgb")
        #print(target_variables_positions_numpy)
       # print(np.unique(target_variables_positions_numpy, axis=0))

        X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions = train_test_split(
            keys_numpy,target_variables_euclidian_distance_numpy,target_variables_positions_numpy,
            test_size=0.2,
            stratify=target_variables_euclidian_distance_numpy,
            random_state=42
        )
    
    return X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions

#### build_X_from_dataframe

In [ ]:
def build_X_from_dataframe(keys_array, df, column_to_take):
    # flatten [[159],[196]...] → [159,196,...]
    keys = keys_array.ravel()

    # number of keys
    N = len(keys)

    # number of time steps per experiment (should be constant)
    example_key = keys[0]
    time_values = df[df["keys"] == example_key]
    T = len(time_values)

    # allocate X
    X = np.zeros((N, T))

    # fill X
    for i, key in enumerate(keys):
        rows = df[df["keys"] == key]
        X[i, :] = rows[column_to_take].values

    return X

#### get_Data_Per_Sensor

In [ ]:
def get_Data_Per_Sensor(userInputData,timeSeriesData_BIG) ->pd.DataFrame:
    df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    sensors_names = np.sort(timeSeriesData_BIG["sensors"].unique())
    dfs_by_sensor = {
        sensor: df_filtered.loc[df_filtered["sensors"]==sensor]
        for sensor in sensors_names
    }
    return dfs_by_sensor


#### class ColumnSliceTransformer

In [ ]:
class ColumnSliceTransformer(BaseEstimator, TransformerMixin):
    """
    Transformer that slices input columns into n_slices parts and returns the slice indexed by slice_idx.
    
    Parameters
    ----------
    n_slices : int
        Total number of slices to split the columns into.
    slice_idx : int
        Index of the slice to select (0-based).
    """
    def __init__(self, n_slices=3, slice_idx=0):
        self.n_slices = n_slices
        self.slice_idx = slice_idx
        
    def fit(self, X, y=None):
        # Number of columns in the input
        self.n_cols_ = X.shape[1]
        # Compute size of each slice
        self.slice_size_ = int(np.ceil(self.n_cols_ / self.n_slices))
        return self

    def transform(self, X):
        # Compute start and end indices for this slice
        start = self.slice_idx * self.slice_size_
        end = min((self.slice_idx + 1) * self.slice_size_, self.n_cols_)
        return X[:, start:end]

In [ ]:
def build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take ="VOC rolling average")->(np.array,np.array):
    dfs_by_sensor = get_Data_Per_Sensor(userInputData,timeSeriesData_BIG)
    X_train = np.empty((len(dfs_by_sensor),X_train_keys.shape[0],330))
    
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_train[i,:,:]= build_X_from_dataframe(X_train_keys,data_per_sensor,column_to_take)
    
    X_test = np.empty((len(dfs_by_sensor),X_test_keys.shape[0],330))
        
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_test[i,:,:]= build_X_from_dataframe(X_test_keys,data_per_sensor,column_to_take)

    return X_train,X_test

#### trimFirstColumns

In [ ]:
def trimFirstColumns(X_train,X_test,columns_to_keep = 15):
    actual_columns_to_keep = abs(-30) + columns_to_keep -1


    X_train =  X_train[:,:,actual_columns_to_keep:]
    X_test  =  X_test[:,:,actual_columns_to_keep:] 
    return X_train,X_test    

#### scaleX

In [ ]:
def scaleX(X_train,X_test,type_of_scaling):
    if type_of_scaling=="normalize":
        for i in range(X_train.shape[0]):
            X_train[i,:,:] =  normalize(X_train[i,:,:])
        for i in range(X_test.shape[0]):
            X_test[i,:,:] =  normalize(X_test[i,:,:])
    elif  type_of_scaling is not None:   
        
        if type_of_scaling=="standard":
            scaler = StandardScaler()
             
        elif type_of_scaling=="minMax":
            scaler = MinMaxScaler()
            
        if type_of_scaling=="robust":
           scaler = RobustScaler()
            
        for i in range(X_train.shape[0]):        
            X_train[i,:,:] = scaler.fit_transform(X_train[i,:,:])
        for i in range(X_test.shape[0]):   
            X_test[i,:,:]  = scaler.transform(X_test[i,:,:])
    
    return X_train,X_test    

#### scaleX rows only 

In [ ]:
#tranpose the rows,scale them and then bring them in the correct position
def scaleXrows(X_train,X_test,type_of_scaling):
    if type_of_scaling=="normalize":
        for i in range(X_train.shape[0]):
            X_train[i,:,:] =  normalize(X_train[i,:,:])
        for i in range(X_test.shape[0]):
            X_test[i,:,:] =  normalize(X_test[i,:,:])
    elif  type_of_scaling is not None:   
        
        if type_of_scaling=="standard":
            scaler = StandardScaler()
             
        elif type_of_scaling=="minMax":
            scaler = MinMaxScaler()
            
        if type_of_scaling=="robust":
           scaler = RobustScaler()
            
        for i in range(X_train.shape[0]):        
            X_train[i,:,:] = np.transpose(scaler.fit_transform(np.transpose(X_train[i,:,:])))
        for i in range(X_test.shape[0]):   
            X_test[i,:,:]  = np.transpose(scaler.transform(np.transpose(X_test[i,:,:])))
    
    return X_train,X_test    

#### takeMeanAndVariance

In [ ]:
def getXMeanAndVariance(X):
    metric_numbers = 2
    X_temp = np.empty((X.shape[0],X.shape[1],metric_numbers))
    for i in range(X_temp.shape[0]):
        for y in range(X_temp.shape[1]):
            X_temp[i,y,0] = X[i,y,:].mean()
            X_temp[i,y,1] = X[i,y,:].var()
         #   X_temp[i,y,2] = np.median(X[i,y,:])
    return X_temp
def takeMeanAndVariance(X_train,X_test):
    
    X_train = getXMeanAndVariance(X_train)
    X_test = getXMeanAndVariance(X_test)

        
        
    return X_train,X_test 

#### take buckets of data

In [ ]:
#shave it to per 15 seconds
def bucketDataX(X,seconds_per_backet):
    X_new =  np.empty((X.shape[0],X.shape[1],X.shape[2] // seconds_per_backet))
    for i in range(X.shape[0]):
        X_new[i,:,:]  =  X[i,:,:].reshape(X.shape[1],X.shape[2] // seconds_per_backet, seconds_per_backet).mean(axis=2)
    return X
def bucketData(X_train,X_test,seconds_per_backet=15):
    X_train = bucketDataX(X_train,seconds_per_backet)
    X_test  = bucketDataX(X_test,seconds_per_backet)
    return X_train,X_test 

#### create_The_Arrays_For_The_Models

In [ ]:

def create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take ="VOC rolling average",keepOpenDoorData =False,statistical_measures=False,
                                     drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None,type_of_scaling='standard'):
    
    userInputData,timeSeriesData_BIG = get_Data_To_Be_Used(userInputData_Original,timeSeriesData_BIG_Original,
                                                           keepOpenDoorData,drop_other_positions,closeDoorTraining_openDoorTest,anyOtherMaskToBeUsed)
    
    X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions = build_X_keys_And_Y_target_Arrays(userInputData)  
    
    X_train,X_test = build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take)
    
    X_train,X_test = scaleXrows(X_train,X_test,type_of_scaling)

    if apply_functions_to_X_data is not None and apply_functions_to_X_data is not np.nan:
       for modify_function in apply_functions_to_X_data:
           if modify_function is not None and modify_function is not np.nan:
               X_train,X_test =  modify_function(X_train,X_test)
    #X_train,X_test =bucketData(X_train,X_test)

   
    return X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG

#### findPCAcomponentsCovered

In [ ]:

    
def findPCAcomponentsCovered(X_train):     
    
    for X_train_per_sensor in X_train:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train_per_sensor)
        # Step 2: Apply PCA
        pca = PCA()
        X_pca = pca.fit_transform(X_scaled)
        
        # Step 3: Explained variance
        explained_variance = pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)
        
        # Only display first 10 components max
        max_components = min(10, len(explained_variance))
        ev_to_plot = explained_variance[:max_components]
        cum_to_plot = cumulative_variance[:max_components]
        
        # Step 4: Bar plot of cumulative explained variance
        plt.figure(figsize=(8,5))
        plt.bar(range(1, max_components + 1), cum_to_plot)
        plt.xlabel('Number of PCA components')
        plt.ylabel('Cumulative explained variance')
        plt.title('Cumulative explained variance (first 10 components)')
        plt.grid(True, axis='y')
        plt.show()
        
        # Step 5: Optimal number of components for ~90% variance
        optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
        print("Optimal number of components to explain ~90% variance:", optimal_components)


#### printPCAcomponents

In [ ]:
def printPCA2components(X_train,Y_train_Euclidians):
    for i in range(X_train.shape[0]):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train[i])
   #     X_scaled = X_train[i]
        # --- Step 2: Reduce to 2 components ---
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        
        # --- Step 3: Scatter plot with hue based on Y_train ---
        plt.figure(figsize=(8,6))
        
        # Using seaborn for easy color scaling
        sns.scatterplot(
            x=X_pca[:,0], 
          #  y=[0]*len(X_pca),              # fake y for 1D PCA
           y=X_pca[:,1], 
            hue=Y_train_Euclidians[:,i],              # color represents Y_train
             palette="coolwarm",         # continuous color palette
            s=80,                      # marker size
            alpha=0.8
        )
        
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.title("2D PCA of X_train colored by Y_train")
        plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.show()

#### printTSNE

In [ ]:
def printTSNE(X_train,Y_train_Euclidians):
    for i in range(X_train.shape[0]):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train[i])
   #     X_scaled = X_train[i]
        # --- Step 2: Reduce to 2 components ---
        pca = TSNE(n_components=2,perplexity=8)
        X_pca = pca.fit_transform(X_scaled)
        
        # --- Step 3: Scatter plot with hue based on Y_train ---
        plt.figure(figsize=(8,6))
        
        # Using seaborn for easy color scaling
        sns.scatterplot(
            x=X_pca[:,0], 
            y=X_pca[:,1], 
            hue=Y_train_Euclidians[:,i],              # color represents Y_train
             palette="coolwarm",         # continuous color palette
            s=80,                      # marker size
            alpha=0.8
        )
        
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.title("2D TSNE of X_train colored by Y_train")
        plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.show()

#### run_grid_search_per_model

In [ ]:
from sklearn.decomposition import KernelPCA


def run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params,statistical_measures = False,type_of_scaling="standard"):
        PCA__n_components = [3,4,6,8,10]
        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']

        results = {}
        if statistical_measures is True:
             pipe = Pipeline([
                 ('model', model)
             ]) 
        else:            
     
             pipe = Pipeline([
             
                ('PCA', PCA()),
                ('model', model)
       
             ])      
        # Build a correct param grid:
        param_grid = {
             **{'model__' + k: v for k, v in params.items()}
         }
        if name not in estimators_with_no_scaling_need and statistical_measures is False:
             param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='neg_mean_squared_error',
            verbose=2
        )
     
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor(X_train, y_train,cv_number,verbose,models,statistical_measures = False,type_of_scaling="standard"):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
     #       print(best_result["name"])
            best_result["score"] = results["score"]
     #       print(best_result["score"])
            best_result["model"] = results["model"]
       #     print(best_result["model"])
            best_result["parameters"] = results["parameters"]
       #     print(best_result["parameters"])
    return best_result


def run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, y_train,cv_number,verbose,models,statistical_measures = False,type_of_scaling="standard"):

    best_results_for_all_sensors = {}

    for i in range(X_train.shape[0]):
        best_results_for_all_sensors[i]  =run_grid_search_find_optimal_model_per_sensor(X_train[i,:,:], y_train[:,i],cv_number,verbose,models,statistical_measures,type_of_scaling)
    print(f"the best models are:{best_results_for_all_sensors}")
    return best_results_for_all_sensors

#### getPredictedValues

In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
def getPredictedValues(X_test,Y_test_Euclidians,best_results_for_all_sensors):

   predict_Euclidians = np.empty((Y_test_Euclidians.shape))
    
    
   for i in range(X_test.shape[0]):
        print(f"For sensor with id={i}")
        predict_Euclidians[:,i] = best_results_for_all_sensors[i]["model"].predict(X_test[i,:,:])
    
   return predict_Euclidians
       

#### trainModels_PrintResults_AndGetPredictedValues

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,models,statistical_measures=False,type_of_scaling="standard"):
    cv_number = 5
    verbose = 0
    
    best_results_for_all_sensors = run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, Y_train_Euclidians,cv_number,verbose,models,statistical_measures,type_of_scaling)
    
    Y_predict_Euclidians = getPredictedValues(X_test,Y_test_Euclidians,best_results_for_all_sensors)
    plantAllBaseSensorsData(Y_test_Euclidians, Y_predict_Euclidians,"Euclidian distance",userInputData)
    return Y_predict_Euclidians

In [ ]:
def grabPositionOfSensors(userInputData):
 # --- SENSOR POINTS (red true, blue/orange predicted) ---
    sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
    point_of_sensors = []
    mask = userInputData["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    for sensors_position_column in sensors_position_columns:
        
        point_of_sensors.append(userInputData.loc[mask,sensors_position_column].iloc[0]) 

    return   point_of_sensors 

#### plot_pred_vs_actual

In [ ]:
def plot_pred_vs_actual(y_test, y_pred, sensor_name, AXIS,userInputData):

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # convert to 2D format
    if y_test.ndim == 1:
        y_test = y_test.reshape(-1,1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1,1)

    # Euclidean error
    EucDis = np.linalg.norm(y_test - y_pred, axis=1).mean()
    
    print(f"\n--- SENSOR {sensor_name} ---")
    print(f"MSE on test set: {mse:.4f}")
    print(f"R^2 on test set: {r2:.4f}")
    print(f"Euclidean distance mean: {EucDis:.4f}")

    # If 2D case
    if y_test.shape[1] == 2:

        plt.figure(figsize=(8,7))

        # --- TRUE (blue) ---
        plt.scatter(y_test[:,0], y_test[:,1], label="True", alpha=0.7, color="blue")
        
        # Print ID next to each true point
        for idx, (x,y) in enumerate(y_test):
            plt.text(x, y, str(idx), color="blue", fontsize=9)

        # --- PREDICTED (orange) ---
        plt.scatter(y_pred[:,0], y_pred[:,1], label="Predicted", alpha=0.7, color="orange")
        
        # Print ID next to each predicted point
        for idx, (x,y) in enumerate(y_pred):
            plt.text(x, y, str(idx), color="orange", fontsize=9)

        point_of_sensors = grabPositionOfSensors(userInputData)
        if point_of_sensors is not None:
            point_of_sensors = np.array(point_of_sensors)

            # TRUE sensor points → BLUE  
            plt.scatter(point_of_sensors[:,0], point_of_sensors[:,1],
                        marker="X", s=200, color="red", label="Sensor Base Points")

        plt.legend()
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.title(f"Sensor {sensor_name} - True vs Predicted")
        plt.grid(True)
        plt.show()

    else:
        # fallback for 1D case
        plt.figure(figsize=(7,6))
        plt.scatter(y_test, y_pred, s=40)
        plt.plot([y_test.min(), y_test.max()], 
                 [y_test.min(), y_test.max()], linestyle='--')
        plt.xlabel("True")
        plt.ylabel("Predicted")
        plt.title(f"{sensor_name} (1D) predictions vs actual")
        plt.grid(True)
        plt.show()
    return -mse, -EucDis, r2   
def plantAllBaseSensorsData(y_test, y_pred,AXIS,userInputData):
    for i in range(y_test.shape[1]):
        AXIS = "Euclidian distances"
        plot_pred_vs_actual(y_test[:,i], y_pred[:,i],i,AXIS,userInputData)


In [ ]:
def trilateration_least_squares(points, radii, tol=5e-2):
    def fun(p):
        x, y = p
        return [np.hypot(x - px, y - py) - r for (px, py), r in zip(points, radii)]

    x0 = [np.mean([px for px,_ in points]),
          np.mean([py for _,py in points])]

    res = least_squares(fun, x0=x0)
    x, y = res.x

    # Residuals close to zero?
    boundary_ok = np.all(np.abs(fun([x, y])) < tol)

    # Check if point lies inside all circles
    inside_ok = all(
        np.hypot(x - px, y - py) <= r + tol
        for (px, py), r in zip(points, radii)
    )

    has_common_point = inside_ok  # this is the correct criterion

    return has_common_point, (x, y)
 

#### findIntersectionOfCircles

In [ ]:
#Take the three points of the sensors
def findIntersectionOfCircles(predict_Euclidians,Y_test_Positions,userInputData):
    
    point_of_sensors = grabPositionOfSensors(userInputData)

    predict_Positions = np.empty((predict_Euclidians.shape[0],2))
    threshold_values =  np.empty((predict_Euclidians.shape[0],1))
    for i in range(predict_Euclidians.shape[0]):
        threshold,point = trilateration_least_squares(point_of_sensors,predict_Euclidians[i,:])
        predict_Positions[i,:] = point
        threshold_values[i] = threshold
    
    MSE, EUCLIDIAN, R2 = plot_pred_vs_actual(Y_test_Positions,predict_Positions,"Final","Point predicted",userInputData)
    return predict_Positions,threshold_values,MSE, EUCLIDIAN, R2

#### PIPELINE

In [ ]:

def runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params ={},apply_functions_to_X_data = [],statistical_measures=False,type_of_scaling="standard"):
   
    (X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,
        timeSeriesData_BIG) = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data,column_to_take,**extra_params,
                                                               statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)
    printPCA2components(X_train,Y_train_Euclidians)

    printTSNE(X_train,Y_train_Euclidians)
    predict_Euclidians = trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,models,type_of_scaling=type_of_scaling)
    print(f"Y_test_Euclidians:{Y_test_Euclidians}")

    print(f"predict_Euclidians:{predict_Euclidians}")
    
    predict_Positions,threshold_values,MSE, EUCLIDIAN, R2 = findIntersectionOfCircles(predict_Euclidians,Y_test_Positions,userInputData)
    
    return (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
    Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,threshold_values,
           MSE,EUCLIDIAN,R2)

In [ ]:
def circle_intersections(c1, r1, c2, r2):
    x0, y0 = c1
    x1, y1 = c2

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    # No intersection
    if d > r1 + r2 or d < abs(r1 - r2) or d == 0:
        return []

    # Compute a, h
    a = (r1**2 - r2**2 + d**2) / (2*d)
    h = math.sqrt(max(0, r1**2 - a**2))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    xs1 = xm + h * dy / d
    ys1 = ym - h * dx / d
    xs2 = xm - h * dy / d
    ys2 = ym + h * dx / d

    return [(xs1, ys1), (xs2, ys2)]

# ------------------------------------------------------------
# Collect intersection points that lie inside all circles
# ------------------------------------------------------------
def intersection_points_of_all_circles(points, radii, tol=1e-6):
    candidates = []

    # pairwise intersections
    for (p1, r1), (p2, r2) in combinations(zip(points, radii), 2):
        pts = circle_intersections(p1, r1, p2, r2)
        candidates.extend(pts)

    # keep only points that lie inside ALL circles
    inside = []
    for x, y in candidates:
        if all(math.hypot(x - px, y - py) <= r + tol
               for (px, py), r in zip(points, radii)):
            inside.append((x, y))

    return inside

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------
def plot_circles_and_intersections(points, radii,i,at):
    fig, ax = plt.subplots(figsize=(6, 6))

    # Draw all circles
    for (px, py), r in zip(points, radii):
        circle = plt.Circle((px, py), r, fill=False, lw=2)
        ax.add_patch(circle)
        ax.plot(px, py, 'ko')  # centers

    # Compute intersections
    ints = intersection_points_of_all_circles(points, radii)

    # Plot intersection points
    for x, y in ints:
        ax.plot(x, y, 'ro', markersize=8)

    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_title("Circles and Intersection Points for row " + str(i) + " at "+at)
    plt.show()

### FILTER FUNCTIONS

In [ ]:
def cutOutliers(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], q25, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], q25, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], q25, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], q25, q75)        

    return X_train,X_test       

In [ ]:
def cutOutliersUpper(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], None, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], None, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersUpperPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], None, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], None, q75)        

    return X_train,X_test       

In [ ]:
def createGradient(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,j,:] = np.gradient(X_train[i,j,:])
            
    for i in range(X_test.shape[0]):
        for j in range(X_test.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_test[i,j,:] = np.gradient(X_test[i,j,:])        

    return X_train,X_test       

### CLASSIFICATION FUNCTIONS

#### getClosestDistances

In [ ]:
def getClosestDistances(userInputData,in_closest_range_number,number_of_sensors):
    print(f"in_closest_range_number:{in_closest_range_number}")
    print(f"number_of_sensors:{number_of_sensors}")
    
    closest_distances = np.empty((in_closest_range_number,number_of_sensors))
    columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
    for i,column_to_grab in enumerate(columns_to_grab):
        closest_distance =  np.sort(userInputData.loc[userInputData[column_to_grab] != None,column_to_grab].unique())[:in_closest_range_number].reshape(1,-1)
        closest_distances[:,i] = closest_distance
    return closest_distances   

#### connect a numpy array to each class

First quantile, use the x axis from -2.95 to -2.5 and y axis from 3.5 to 2.5,
Second quantile, use the x axis from -1.5 to -0.5 and y axis from 3.5 to 2.5,
Third quantile, use the x axis from -2.95 to -2.5 and y axis from 1.5 to 0.5,
First quantile, use the x axis from -1.5 to -0.5 and y axis from 3.5 to 2.5,


In [ ]:
def connectPositionToClass():
    pass

#### build_X_keys_And_Y_target_Arrays

In [ ]:

def build_X_keys_And_Y_target_Arrays_Classification(userInputData,in_closest_range_number,type_of_classification="close-far",closeDoorTraining_openDoorTest = False)->(np.array,np.array):
  
    if closeDoorTraining_openDoorTest :
         pass

    else:
        keys = np.array(userInputData.index)
        keys_numpy = keys.reshape(-1,1)

        columns_to_pass = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                                   for id in range(3) ]
        column_size = len(columns_to_pass)
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_euclidian_distance_numpy = target_variables.reshape(-1,column_size)
     
    
        
        columns_to_pass = ["x-y axis"]
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_positions_numpy = np.stack(target_variables[:,0])  
        print(target_variables_positions_numpy)
        #create close classes
        classes = target_variables_euclidian_distance_numpy.copy()
        number_of_sensors = target_variables_euclidian_distance_numpy.shape[1]

        if type_of_classification == "close-far":

            closest_distances = getClosestDistances(userInputData,in_closest_range_number,number_of_sensors)
            for i in range(classes.shape[1]):
                classes[:,i] =np.where(np.isin(close_condition[:,i],closest_distances[:,i]),1,0)
        #create 4 difference classes,but keep the shape of outward classes the same        
        elif type_of_classification == "four parts":
             for i in range(classes.shape[1]):
                 pass
                 #first quartile of the room
                 # mask = ((classes[:, 0] > -3) & (classes[:, 0] < -2)) & ((classes[:, 0] > -3) & (classes[:, 0] < -2))

                 # classes = np.where(mask[:, None], [99, 99], a) 
                 # classes[:,i] =np.where(
        X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions,Y_train_classes,Y_test_classes = train_test_split(
            keys_numpy,target_variables_euclidian_distance_numpy,target_variables_positions_numpy,classes,
            test_size=0.166,
            stratify=classes,
            random_state=42
        )
    
    return X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions,Y_train_classes,Y_test_classes

#### create_The_Arrays_For_The_Models

In [ ]:

def create_The_Arrays_For_The_Models_Close_Class(userInputData_Original,timeSeriesData_BIG_Original,in_closest_range_number,apply_functions_to_X_data = [],
                                     column_to_take ="VOC rolling average",keepOpenDoorData =False,statistical_measures=False,
                                     drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None):
    
    userInputData,timeSeriesData_BIG = get_Data_To_Be_Used(userInputData_Original,timeSeriesData_BIG_Original,
                                                           keepOpenDoorData,drop_other_positions,closeDoorTraining_openDoorTest,anyOtherMaskToBeUsed)
    
    (X_train_keys,X_test_keys,Y_train_Euclidians,
         Y_test_Euclidians,Y_train_Positions,Y_test_Positions,
            Y_train_close,Y_test_close) = build_X_keys_And_Y_target_Arrays_Classification(userInputData,in_closest_range_number)  
    
    X_train,X_test = build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take)
    X_train,X_test = normalizeX(X_train,X_test)
    if statistical_measures:
        X_train,X_test = takeMeanAndVariance(X_train,X_test)

    if apply_functions_to_X_data is not None and apply_functions_to_X_data is not np.nan:
       for modify_function in apply_functions_to_X_data:
           if modify_function is not None and modify_function is not np.nan:
               X_train,X_test =  modify_function(X_train,X_test)
 
   
    return X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions,Y_train_close,Y_test_close, userInputData,timeSeriesData_BIG

#### run_grid_search_per_model

In [ ]:

def run_grid_search_per_model_Classifier(X_train, y_train,cv_number,verbose,name,model,params,sensors_numbers,sensor_data_to_grab):
        PCA__n_components = [2,3,4,5,6,8,10]

        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']
        print(f"sensors_numbers:{sensors_numbers}")
        print(f"sensor_data_to_grab:{sensor_data_to_grab}")
        results = {}
    
        if name in estimators_with_no_scaling_need:
             pipe = Pipeline([
                 ('columnslice',ColumnSliceTransformer( n_slices=sensors_numbers, slice_idx=sensor_data_to_grab)),
                 ('model', model)
             ]) 
        else:
             pipe = Pipeline([
                 ('columnslice',ColumnSliceTransformer( n_slices=sensors_numbers, slice_idx=sensor_data_to_grab)),
                 ('scaler', StandardScaler()),
                 ('PCA', PCA()),
                 ('model', model)
             ])
        # Build a correct param grid:
        param_grid = {
            **{'model__' + k: v for k, v in params.items()}
        }
        if name not in estimators_with_no_scaling_need:
            param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='accuracy',
            verbose=2
        )
     
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor_Classifier(X_train, y_train,cv_number,verbose,models,sensors_numbers=3,sensor_data_to_grab=0):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -9999,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model_Classifier(X_train, y_train,cv_number,verbose,name,model,params,sensors_numbers,sensor_data_to_grab)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
     #       print(best_result["name"])
            best_result["score"] = results["score"]
     #       print(best_result["score"])
            best_result["model"] = results["model"]
       #     print(best_result["model"])
            best_result["parameters"] = results["parameters"]
       #     print(best_result["parameters"])
    return best_result


def run_grid_search_find_optimal_model_per_sensor_for_all_sensors_Classifier(X_train, y_train,cv_number,verbose,models,sensors_numbers):

    best_results_for_all_sensors = {}
    
    for sensor_data_to_grab in range(sensors_numbers):
        print(f"for sensor with id:{sensor_data_to_grab}")
        best_results_for_all_sensors[sensor_data_to_grab]  =run_grid_search_find_optimal_model_per_sensor_Classifier(X_train, y_train[:,sensor_data_to_grab],cv_number,verbose,models,sensors_numbers,sensor_data_to_grab)

    return best_results_for_all_sensors

In [ ]:
def find_best_stacked_regression_Pre_Given(X_train, y_train,cv_number,verbose,models,best_result_per_sensor):
    
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {} 
    }
    print("For Stacking Classifier")
    for final_estimator_name, (final_estimator_model, final_estimator_params) in models.items():
            
        base_estimators = [(str(sensor_names),base_best_result['model']) 
                               for sensor_names,base_best_result in best_result_per_sensor.items()]
    
       # print(f"base_estimators{base_estimators}")
        stacked_Estimator = StackingClassifier(
            estimators=base_estimators,
            final_estimator=final_estimator_model,  # we will tune this
            passthrough=True           # include original features if you want
        )
        param_grid = {
            **{'final_estimator' +'__' + k: v for k, v in final_estimator_params.items()}
        }
  
        stacked_grid = GridSearchCV(stacked_Estimator,param_grid,cv=cv_number,verbose=verbose,n_jobs=-1,scoring='accuracy')
        stacked_grid.fit(X_train, y_train)
    
        
        if stacked_grid.best_score_ > best_result["score"]:
                best_result["name"]  = final_estimator_name
                best_result["model"] = stacked_grid.best_estimator_
                best_result["parameters"] = stacked_grid.best_params_
                best_result["score"] = stacked_grid.best_score_ 
    return  best_result           

#### votting Classifier

In [ ]:
def voteBestClassifierCreate(X_train, y_train,best_result_per_sensor,AXIS=''):

    base_estimators = [(str(sensor_names),base_best_result['model']) 
                               for sensor_names,base_best_result in best_result_per_sensor.items()]
    VoterClassifier = VotingClassifier(base_estimators)
    if AXIS == "X AXIS":
        column = 0 
    elif AXIS == "Y AXIS":
        column = 1 
    else:
        column = 0
    VoterClassifier = VoterClassifier.fit(X_train, y_train)
    return VoterClassifier

#### getPredictedValues

In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
def getPredictedValuesClassified(X_test,Y_test,best_results_for_all_sensors,sensors_numbers,is_Stacked = False):


   if (is_Stacked is False): 
       predict_Y = np.empty((Y_test.shape[0],sensors_numbers), dtype=int)
        
       for i in range(sensors_numbers):
            print(f"For sensor with id={i}")
            print(best_results_for_all_sensors)
            predict_Y[:,i] = best_results_for_all_sensors[i]["model"].predict(X_test)
   else:
        predict_Y = np.empty((Y_test.shape[0]))

        predict_Y = best_results_for_all_sensors.predict(X_test)

   return predict_Y
       

#### STACK_X_TRAIN_X_TEST

In [ ]:
# MAKE THE X_TRAIN AND X_TEST 3 DIMENSION STACKED INTO THE COLUMNS


def stack_X_train_And_X_Test(X_train,X_test):
    sensors_numbers = X_train.shape[0]
    X_train_Stacked = np.column_stack(tuple(X_train))
    X_test_Stacked  = np.column_stack(tuple(X_test))
    return X_train_Stacked,X_test_Stacked,sensors_numbers

#### trainModels_PrintResults_AndGetPredictedValues

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues_Classification(userInputData, X_train,X_test,Y_train,Y_test,models):
    cv_number = 5
    verbose = 0
    X_train,X_test,sensors_numbers = stack_X_train_And_X_Test(X_train,X_test)
    print(sensors_numbers)
    print(f"sensors_numbers:{sensors_numbers}")

    best_results_for_all_sensors = {}

    best_results_for_all_sensors = run_grid_search_find_optimal_model_per_sensor_for_all_sensors_Classifier(X_train, Y_train,cv_number,verbose,models,sensors_numbers)
 
    
    Y_predict_Base = getPredictedValuesClassified(X_test,Y_test,best_results_for_all_sensors,sensors_numbers,is_Stacked = False)
    
    for i,Y_predict in enumerate(np.transpose(Y_predict_Base)):
         
         plotConfusionMatrix(Y_test[:,i], Y_predict_Base[:,i])
         model = best_results_for_all_sensors[i]["model"]
         getScores(Y_test[:,i], Y_predict_Base[:,i],model,X_train,Y_train[:,i])
        
   # plantAllBaseSensorsData(Y_test, Y_predict_Base,"X-Y AXIS",userInputData,Label_Encoder,sensors_numbers)

    return Y_predict_Base,best_results_for_all_sensors

#### trainModels_PrintResults_AndGetPredictedValues_Stacked

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues_Stacked_Classification(userInputData,best_results_for_all_sensors,X_train,X_test,Y_train,Y_test,models):
    cv_number = 5
    verbose = 0
    best_results_stacked = {}
    
    X_train,X_test,sensors_numbers = stack_X_train_And_X_Test(X_train,X_test)
    print(f"sensors_numbers:{sensors_numbers}")
    print(f"best_results_for_all_sensors:{best_results_for_all_sensors}")
  
    best_results_stacked = voteBestClassifierCreate(X_train, Y_train,best_results_for_all_sensors,AXIS='')
 
    
    
    Y_predict_Stacked = getPredictedValuesClassified(X_test,Y_test,best_results_stacked,sensors_numbers,is_Stacked = True)
    
     
    plotConfusionMatrix(Y_test, Y_predict_Stacked)
    model = best_results_stacked
    accuracy,precision,recall =getScores(Y_test, Y_predict_Stacked,model,X_train,Y_train)
  #  mse, EucDis, r2  = plot_pred_vs_actual(Y_test, Y_predict_Stacked, None, "X-Y AXIS",userInputData,Label_Encoder)

    return Y_predict_Stacked,best_results_stacked, accuracy,precision,recall

#### PLOT CONFUSION MATRIX

In [ ]:
def plotConfusionMatrix(y_test, y_pred):
    # encoder used earlier
 
    
    cm = confusion_matrix(y_test, y_pred)
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
    )
    
    disp.plot(cmap='Blues', values_format='d')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

#### GET SCORES

In [ ]:
def getScores(y_test, y_pred,model,X_train,y_train):
    
    y_train_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    
    print(f"Training accuracy: {train_accuracy:.4f}")
    
    print(f"y_test:{y_test}")
    print(f"y_pred:{y_pred}")
    accuracy = accuracy_score(y_test, y_pred)
   
    precision = precision_score(
        y_test, y_pred,
        average='weighted'   # or 'macro', 'micro'
    )
    
    recall = recall_score(
        y_test, y_pred,
        average='weighted'
    )
    
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    return accuracy,precision,recall

#### CLASSIFICATION PIPELINE

In [ ]:

def runPipelineClassiciation(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,in_closest_range_number,extra_params ={},apply_functions_to_X_data = [],statistical_measures=False):
   
    (X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions,
     Y_train_close,Y_test_close, userInputData,timeSeriesData_BIG) = create_The_Arrays_For_The_Models_Close_Class(userInputData_Original,timeSeriesData_BIG_Original,in_closest_range_number,apply_functions_to_X_data,column_to_take,**extra_params,statistical_measures=statistical_measures)
   # printPCA1components(X_train,Y_train_Euclidians)

  #  printPCA2components(X_train,Y_train_Euclidians)

    

    Y_predict_Base,best_model_per_second = trainModels_PrintResults_AndGetPredictedValues_Classification(userInputData, X_train,X_test,Y_train_close,Y_test_close,models)
    # output of the based models to stacked models
   # Y_predict_Stacked,best_model, accuracy,precision,recall = trainModels_PrintResults_AndGetPredictedValues_Stacked_Classification(userInputData,best_model_per_second, X_train,X_test,Y_train_close,Y_test_close,models)

    return (Y_predict_Base,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
    Y_train_Positions,Y_test_Positions,Y_train_close,Y_test_close, userInputData,timeSeriesData_BIG,
         )

# FIND THE VARIABLES 

## CREATE DATAFRAMES AND FUNCTION TO PRINT

In [ ]:
#functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
#functions_to_apply_Gradient = [None,createGradient]
functions_to_apply_Outliers = [None,cutOutliers]
functions_to_apply_Gradient = [None]
functions_to_apply_Outliers_names = [X.__name__ if X is not None else X for X in functions_to_apply_Outliers]
functions_to_apply_Gradient_names = [X.__name__ if X is not None else X for X in functions_to_apply_Gradient]

column_to_take =["VOC smoothed"]
room_cases = ["Only closed","All data"]
multi_index = pd.MultiIndex.from_product([room_cases,column_to_take,functions_to_apply_Gradient_names,functions_to_apply_Outliers_names],names= ["room cases" ,"column to take","Gradient functions","Outlier functions"])
for idx in multi_index:
    print(idx)

In [ ]:
columns = ["MSE","EUCLIDIAN","R2"]
parametersResultsGathered = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGathered

#### loopTroughFunctions

In [ ]:
def loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,
                            statistical_measures=False,functions_to_apply_Outliers=functions_to_apply_Outliers,functions_to_apply_Gradient=functions_to_apply_Gradient,type_of_scaling="standard"):
  
    for function_Gradient in functions_to_apply_Gradient:

        for function_Outlier in functions_to_apply_Outliers:
            outlier_name = function_Outlier.__name__ if function_Outlier is not None else np.nan
            gradient_name = function_Gradient.__name__ if function_Gradient is not None else np.nan
            print("outlier_name "+str(outlier_name))
            print("gradient_name "+str(gradient_name))


            apply_functions_to_X_data = [function_Outlier,function_Gradient]

            (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
    Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,threshold_values,
           MSE,EUCLIDIAN,R2) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,
                                                  column_to_take,extra_params,apply_functions_to_X_data,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)
            print(MSE,EUCLIDIAN,R2)
            idx = (room_case, column_to_take, gradient_name, outlier_name)
            print(idx)
            parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]] = [MSE, EUCLIDIAN, R2]
            print(parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]])
    return parametersResultsGathered      
            # MSE,EUCLIDIAN,R2 the values i want to insert 

#### loopTroughFunctionsClassification

In [ ]:
def loopTroughFunctionsClassification(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered,
                            statistical_measures=False,functions_to_apply_Outliers=functions_to_apply_Outliers,functions_to_apply_Gradient=functions_to_apply_Gradient):
   

    
    for function_Gradient in functions_to_apply_Gradient:

        for function_Outlier in functions_to_apply_Outliers:
            outlier_name = function_Outlier.__name__ if function_Outlier is not None else np.nan
            gradient_name = function_Gradient.__name__ if function_Gradient is not None else np.nan
            print("outlier_name "+str(outlier_name))
            print("gradient_name "+str(gradient_name))


            apply_functions_to_X_data = [function_Outlier,function_Gradient]

            (Y_predict_Base,Y_predict_Stacked,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
            Y_train_Positions,Y_test_Positions,Y_train_close,Y_test_close, userInputData,timeSeriesData_BIG,
            accuracy,precision,recall) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,
                                                  column_to_take,extra_params,apply_functions_to_X_data,statistical_measures=statistical_measures)
            print(MSE,EUCLIDIAN,R2)
            idx = (room_case, column_to_take, gradient_name, outlier_name)
            print(idx)
            parametersResultsGathered.loc[idx, ["accuracy","precision","recall"]] = [MSE, EUCLIDIAN, R2]
            print(parametersResultsGathered.loc[idx, ["accuracy","precision","recall"]])
            
    return parametersResultsGathered      
            # MSE,EUCLIDIAN,R2 the values i want to insert 

In [ ]:

columns = ["accuracy","precision","recall"]
parametersResultsGathered = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGathered

#### grabClosestDataPerSensor

In [ ]:
def grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=True):
    #grab only for most recent roomsetup
    mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    userInputData = userInputData_Original.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData.index)].copy()
    
    #grab the data closest to each sensor
    
    #start from just an empty boolean expression and add the conditions
    mask = pd.Series(False, index=userInputData.index)
    columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
    for column_to_grab in columns_to_grab:
        
        smallest_distances = np.sort(userInputData.loc[userInputData[column_to_grab] != None,column_to_grab].unique())
        print(smallest_distances[:number_of_distances_to_grab])
        if grab_closest:
            mask = mask | (userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
        else:
            mask = mask | (~userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
            
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
    return userInputData,timeSeriesData_BIG

#### grab_from_list

In [ ]:
def grab_from_list(userInputData_Original,timeSeriesData_BIG_Original):
    keys_to_keep = [181,186,237,46,97,113,
                    98,182,247,213,185,
                    118,150,198,171,92,125,
                    124,130,191,88,149,
                    159,201,250,109,142,174,216,234,
                    166,175,235,160,143]
    mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    userInputData = userInputData_Original.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData.index)].copy()

    mask = userInputData.index.isin(keys_to_keep)
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
    return userInputData,timeSeriesData_BIG

#### printDataGrabbed

In [ ]:
def printDataGrabbed(userInputData,timeSeriesData_BIG,column_to_take = "VOC smoothed",groupby=["room","are-doors-opened","x axis","y axis"] ,type_of_other_grouping = ["door-opened","position"]):
    print("All data")
    
    room_other_grouping = userInputData_Original.groupby(groupby).groups

    printDataBasedOnDate(column_to_take,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping)
    print("per sensor")
    sensors = timeSeriesData_BIG["sensors"].unique()
    for sensor in sensors:
        print(sensor)
        column_to_take = "VOC smoothed"
        plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take,sensor)

## RUN THE FUNCTIONS

### GRAB THE REDUCED DATA 

In [ ]:
userInputData_Original.tail(20)

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
userInputData,timeSeriesData_BIG = grab_from_list(userInputData_Original,timeSeriesData_BIG_Original)
userInputData.head(30)

In [ ]:
userInputData_Original["room"].unique()

In [ ]:
# we will keep the closest places 
n = 2
places_to_keep = []
columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
for column_to_grab in columns_to_grab:
    smallest_distances = np.sort(userInputData_Original.loc[userInputData_Original[column_to_grab] != None,column_to_grab].unique())
    print(smallest_distances[:n])
    
    

### RUN THE REGRESSION MODELS

### RUN VOC SMOOTHED NORMALIZED

#### set constant variables to be used


In [ ]:
type_of_scaling = "normalize"
statistical_measures=False
column_to_take  = "VOC smoothed"

def set_keepOpenDoorData(room_case):
    if room_case=="Only closed":
        extra_params = { "keepOpenDoorData" :False,'drop_other_positions':True}
    if room_case=="All data":     
        extra_params = { "keepOpenDoorData" :True,'drop_other_positions':True}
    return extra_params

#### grab_from_list CHOSEN ONES

In [ ]:

room_case = "All data"
extra_params = set_keepOpenDoorData(room_case)

userInputData,timeSeriesData_BIG = grab_from_list(userInputData_Original,timeSeriesData_BIG_Original)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
parametersResultsGathered.loc[room_case]

#### RUN ALL

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB FIRST Distances

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)

number_of_distances_to_grab = 1

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)
parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB Second Distances

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
number_of_distances_to_grab = 2

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)


parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

### RUN VOC SMOOTHED STANDARD SCALER

#### set constant variables to be used


In [ ]:
type_of_scaling = "standard"
statistical_measures=False
column_to_take  = "VOC smoothed"
#extra_params = { "keepOpenDoorData" :True,'drop_other_positions':False}
def set_keepOpenDoorData(room_case):
    if room_case=="Only closed":
        extra_params = { "keepOpenDoorData" :False,'drop_other_positions':True}
    if room_case=="All data":     
        extra_params = { "keepOpenDoorData" :True,'drop_other_positions':True}
    return extra_params

#### grab_from_list CHOSEN ONES

In [ ]:

room_case = "All data"
extra_params = set_keepOpenDoorData(room_case)
userInputData,timeSeriesData_BIG = grab_from_list(userInputData_Original,timeSeriesData_BIG_Original)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### RUN ALL

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB FIRST Distances

In [ ]:
column_to_take = "VOC smoothed"
room_case = "Only closed"
number_of_distances_to_grab = 1

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)
parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB Second Distances

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
number_of_distances_to_grab = 2

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)


parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB EVERYTHING THAN THE FIRST Distances

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
number_of_distances_to_grab = 1

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=False)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

parametersResultsGathered.loc[room_case]

#### GRAB EVERYTHING THAN THE SECONDS Distances

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
number_of_distances_to_grab = 2

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=False)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

parametersResultsGathered.loc[room_case]

### RUN VOC SMOOTHED ROBUST

#### set constant variables to be used


In [ ]:
type_of_scaling = "robust"
statistical_measures=False
column_to_take  = "VOC smoothed"
extra_params = { "keepOpenDoorData" :True,'drop_other_positions':False}
def set_keepOpenDoorData(room_case):
    if room_case=="Only closed":
        extra_params = { "keepOpenDoorData" :False,'drop_other_positions':True}
    if room_case=="All data":     
        extra_params = { "keepOpenDoorData" :True,'drop_other_positions':True}
    return extra_params

#### grab_from_list CHOSEN ONES

In [ ]:

room_case = "All data"
extra_params = set_keepOpenDoorData(room_case)
userInputData,timeSeriesData_BIG = grab_from_list(userInputData_Original,timeSeriesData_BIG_Original)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### RUN ALL

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB FIRST Distances

In [ ]:
column_to_take = "VOC smoothed"
room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
number_of_distances_to_grab = 1
statistical_measures = False

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)
parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB Second Distances

In [ ]:

room_case = "Only closed"
extra_params = set_keepOpenDoorData(room_case)
number_of_distances_to_grab = 2

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)


parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB EVERYTHING THAN THE FIRST Distances

column_to_take = "VOC smoothed"
room_case = "Only closed"
number_of_distances_to_grab = 1

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=False)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

parametersResultsGathered.loc[room_case]

#### GRAB EVERYTHING THAN THE SECONDS Distances

column_to_take = "VOC smoothed"
room_case = "Only closed"
number_of_distances_to_grab = 2
statistical_measures = False

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=False)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

parametersResultsGathered.loc[room_case]

### RUN VOC SMOOTHED minmax

#### set constant variables to be used


In [ ]:
type_of_scaling = "minMax"
statistical_measures=False
column_to_take  = "VOC smoothed"
extra_params = { "keepOpenDoorData" :True,'drop_other_positions':False}

#### grab_from_list CHOSEN ONES

In [ ]:
room_case = "All data"
extra_params = set_keepOpenDoorData(room_case)

userInputData,timeSeriesData_BIG = grab_from_list(userInputData_Original,timeSeriesData_BIG_Original)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### RUN ALL

In [ ]:

room_case = "Only closed"
parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB FIRST Distances

In [ ]:
column_to_take = "VOC smoothed"
room_case = "Only closed"
number_of_distances_to_grab = 1
statistical_measures = False

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)
parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB Second Distances

In [ ]:

room_case = "Only closed"
number_of_distances_to_grab = 2

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab)


parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,
                                                column_to_take,room_case,extra_params,parametersResultsGathered,
                                                statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

In [ ]:
parametersResultsGathered.loc[room_case]

#### GRAB EVERYTHING THAN THE FIRST Distances

column_to_take = "VOC smoothed"
room_case = "Only closed"
number_of_distances_to_grab = 1

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=False)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

parametersResultsGathered.loc[room_case]

#### GRAB EVERYTHING THAN THE SECONDS Distances

column_to_take = "VOC smoothed"
room_case = "Only closed"
number_of_distances_to_grab = 2
statistical_measures = False

userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=False)

parametersResultsGathered = loopTroughFunctions(userInputData,timeSeriesData_BIG,models,column_to_take,
                                                room_case,extra_params,parametersResultsGathered,statistical_measures=statistical_measures,type_of_scaling=type_of_scaling)

parametersResultsGathered.loc[room_case]

## END 

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
    key=lambda col: col.abs()
)
df_sorted

# FEATURE SEARCH 

## FUNCTIONS TO USE 

In [ ]:
room_other_grouping = userInputData_Original.groupby(["room","are-doors-opened","x axis","y axis"]).groups
type_of_other_grouping = ["door-opened","position"]
column_to_take = "VOC smoothed"
userInputData_Original_temp = userInputData_Original.loc[(userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0') & (userInputData_Original["are-doors-opened"]!="on")] 
printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

In [ ]:
def grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab=0,grab_closest=True):
    #grab only for most recent roomsetup
    mask = userInputData_Original["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    userInputData = userInputData_Original.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData.index)].copy()
    
    #grab the data closest to each sensor
    
    #start from just an empty boolean expression and add the conditions
    if number_of_distances_to_grab > 0:

        mask = pd.Series(False, index=userInputData.index)
    elif number_of_distances_to_grab == 0:  
        mask = pd.Series(True, index=userInputData.index)
        
    columns_to_grab= ["Euclidian distance to Id="+str(i)+":BME680:breathVocEquivalent" for i in range(3)]
    for column_to_grab in columns_to_grab:
        
        smallest_distances = np.sort(userInputData.loc[userInputData[column_to_grab] != None,column_to_grab].unique())
        if grab_closest:
            mask = mask | (userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
        else:
            mask = mask | (~userInputData[column_to_grab].isin(smallest_distances[:number_of_distances_to_grab]))
            
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
    return userInputData,timeSeriesData_BIG

In [ ]:
def scaleX(X_train,X_test,type_of_scaling):
    for i in range(X_train.shape[0]):
        if type_of_scaling == "standardScaler":
            
            scaler = StandardScaler()
            X_train[i,:,:] = scaler.fit_transform(X_train[i,:,:])
            X_test[i,:,:]  =  scaler.transform(X_test[i,:,:])
        if type_of_scaling == "normalize":    
            X_train[i,:,:] = normalize(X_train[i,:,:])
            X_test[i,:,:]  = normalize(X_test[i,:,:])
    return X_train,X_test

In [ ]:
def create_indices(Y_whole):
    indices = {}
    for i in range(Y_whole.shape[1]):
        indices[i] = {label: np.where(Y_whole[:,i] == label)[0] for label in np.unique(Y_whole[:,i])}
    return indices   

In [ ]:
def create_pos_indices(Y_whole):
    unique_rows = np.unique(Y_whole, axis=0)

    indices = {}
    for row in unique_rows:
        row_key = tuple(row)  # convert to hashable key
        mask = np.all(Y_whole == row, axis=1)
        indices[row_key] = np.where(mask)[0]

    return indices 

In [ ]:
def grab_X_whole_Y_whole(column_to_take,number_of_distances_to_grab,grab_closest,type_of_scaling="None",
                         userInputData_Original=userInputData_Original,timeSeriesData_BIG_Original=timeSeriesData_BIG_Original):

    userInputData,timeSeriesData_BIG = grabClosestDataPerSensor(userInputData_Original,timeSeriesData_BIG_Original,number_of_distances_to_grab,grab_closest=True)
    X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData,timeSeriesData_BIG,apply_functions_to_X_data = [],
                                         column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)
    X_train,X_test = scaleX(X_train,X_test,type_of_scaling)
    X_whole = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],X_train.shape[2]))
    for i,_ in enumerate(X_whole):
        X_whole[i,:,:] = np.vstack((X_train[i,:,:],X_test[i,:,:]))
   
    
    Y_whole = np.empty((Y_train_Euclidians.shape[0]+Y_test_Euclidians.shape[0],Y_train_Euclidians.shape[1]))
    Y_whole = np.vstack((Y_train_Euclidians,Y_test_Euclidians))  
    indices = create_indices(Y_whole)
    
    Y_whole = np.empty((Y_train_Positions.shape[0]+Y_test_Positions.shape[0],Y_train_Positions.shape[1]))
    Y_whole = np.vstack((Y_train_Positions,Y_test_Positions))  
    pos_indices = create_pos_indices(Y_whole)
    return X_whole,Y_whole,indices,pos_indices

## FUNCTIONS TO USE PLOTTING

In [ ]:
def boxplot(data,sensor_code,showfliers=True,euclidian_distance=None):
    data_flatten = data.ravel()
    plt.figure(figsize=(20, 3))  # ultra-wide & flat

    sns.boxplot(x=data_flatten,showfliers=showfliers)
    
    title = f"Box plot for sensor with id={i}"
    if showfliers is False:
        add_title = f",with no outliers"
        title = title + add_title 
    if euclidian_distance is not None:
        add_title = f",at euclidian distance {target_variable}"
        title = title + add_title
    plt.title(title)

    plt.show()

In [ ]:
def histplot(data,sensor_code,euclidian_distance=None):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
   
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
        
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    title = f"Hist plots for sensor with id={i}"
    
    if euclidian_distance is not None:
        add_title = f",at euclidian distance {target_variable}"
        title = title + add_title
    # Figure-wide title
    fig.suptitle(title, fontsize=16, y=1.02)
        
    plt.show()        

In [ ]:
def heatmapMatrixAll(all_dfs, identity):
    fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)

    for i in all_dfs.keys():
        sns.heatmap(
            all_dfs[i],
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            #square=True,
            annot_kws={"size": 9},
            cbar=i == (len(all_dfs.keys()) - 1),  # show colorbar only on last plot
            cbar_kws={"label": identity},
            ax=axes[i]
        )
        axes[i].set_title(f"Sensor ID = {i}")

    fig.suptitle(f"{identity} Matrix Heatmaps", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def CorrelationMatrixPerDistance(column_to_use,X_whole,indices):
    print(f"use {column_to_use}")
    all_dfs ={}
    for sensor,groups_based_on_distance in indices.items():
        mean_df=pd.DataFrame(data=np.nan,index=groups_based_on_distance,columns=["mean"])
        for target_variable,group_indexes in groups_based_on_distance.items():
                idx = np.array(group_indexes)
                array_to_get = X_whole[sensor,idx]
                corr_matrix = np.corrcoef(array_to_get)
                mean_of_corr_matrix = np.mean(corr_matrix)
                mean_df.loc[target_variable,"mean"] = mean_of_corr_matrix
        all_dfs[sensor] = mean_df
    #all_dfs    
    heatmapMatrixAll(all_dfs, "Correlation")    
    return all_dfs

In [ ]:
def DTWscoreSamePositionDifferentSensors(column_to_use,X_whole,pos_indices):
    
    print(f"use {column_to_use}")
    all_dfs ={}
    #do it for every sensor
    for sensor in range(X_whole.shape[0]):
        mean_df=pd.DataFrame(data=np.nan,index=pos_indices.keys(),columns=[i for i in range(X_whole.shape[0])])
        #do it for both itself and pair of the other sensors
        for paired_sensor in mean_df.columns:
            for position,group_indexes in pos_indices.items():
                    dtw_array = np.empty((group_indexes.shape[0],group_indexes.shape[0]))
                    for i,index_row in enumerate(group_indexes):
                        for j,index_column in enumerate(group_indexes):
                            
                            timeseries_index = to_time_series(X_whole[sensor,index_row,:])
                            timeseries_column = to_time_series(X_whole[paired_sensor,index_column,:])
                            dtw_score = dtw(timeseries_index, timeseries_column)
                            dtw_array[i,j] = dtw_score
                    mean_value =  np.mean(dtw_array)       
                    mean_df.loc[position,paired_sensor]  =  mean_value
        all_dfs[sensor] = mean_df       
    heatmapMatrixAll(all_dfs, "Dynamic time wrapping (DTW) mean values")           
    return all_dfs


## VOC SMOOTHED NO DATA CHANGE

### GRAB THE DATA

In [ ]:
    print("print Data Grabbed")
    room_other_grouping = userInputData_Original.groupby(["room","are-doors-opened","x axis","y axis"]).groups
    type_of_other_grouping = ["door-opened","position"]
    column_to_take = "VOC smoothed"
    printDataBasedOnDate(column_to_take,userInputData_Original,timeSeriesData_BIG_Original,room_other_grouping,type_of_other_grouping)

In [ ]:
column_to_take = "VOC smoothed"
number_of_distances_to_grab = 0
grab_closest = False
type_of_scaling="None"
X_whole,Y_whole,indices,pos_indices =grab_X_whole_Y_whole(column_to_take,number_of_distances_to_grab,grab_closest,type_of_scaling="None",
                         userInputData_Original=userInputData_Original,timeSeriesData_BIG_Original=timeSeriesData_BIG_Original)



In [ ]:
X_whole

In [ ]:
Y_whole

In [ ]:
indices

In [ ]:
pos_indices

### DATA DISTRIBUTION PLOT FOR ALL DATA

In [ ]:
for i,data in enumerate(X_whole):

    boxplot(data,i,showfliers=True,euclidian_distance=None)

In [ ]:
for i,data in enumerate(X_whole):

    boxplot(data,i,showfliers=False,euclidian_distance=None)

In [ ]:
for sensor_id, data in enumerate(X_whole):
     histplot(data,sensor_id,euclidian_distance=None)


### DATA DISTRIBUTION PLOT FOR EVERY DIFFERENT DISTANCE

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            boxplot(data,i,showfliers=True,euclidian_distance=target_variable)

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            boxplot(data,i,showfliers=False,euclidian_distance=target_variable)

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            histplot(data,i,euclidian_distance=None)

### EXPLORE CORROLEATION BETWEEN SAME EUCLIDIAN DISTANCE AND DTW BETWEEEN SAME POSITION

In [ ]:
pos_indices

In [ ]:

CorrelationMatrixPerDistance("smoothed VOC",X_whole,indices)

In [ ]:
DTWscoreSamePositionDifferentSensors("smoothed VOC",X_whole,pos_indices)

## VOC SMOOTHED standardScaler

### GRAB THE DATA

In [ ]:
column_to_take = "VOC smoothed"
number_of_distances_to_grab = 0
grab_closest = False
type_of_scaling="standardScaler"
X_whole,Y_whole,indices,pos_indices =grab_X_whole_Y_whole(column_to_take,number_of_distances_to_grab,grab_closest,type_of_scaling="None",
                         userInputData_Original=userInputData_Original,timeSeriesData_BIG_Original=timeSeriesData_BIG_Original)



In [ ]:
X_whole

In [ ]:
Y_whole

In [ ]:
indices

In [ ]:
pos_indices

### DATA DISTRIBUTION PLOT FOR ALL DATA

In [ ]:
for i,data in enumerate(X_whole):

    boxplot(data,i,showfliers=True,euclidian_distance=None)

In [ ]:
for i,data in enumerate(X_whole):

    boxplot(data,i,showfliers=False,euclidian_distance=None)

In [ ]:
for sensor_id, data in enumerate(X_whole):
     histplot(data,sensor_id,euclidian_distance=None)


### DATA DISTRIBUTION PLOT FOR EVERY DIFFERENT DISTANCE

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            boxplot(data,i,showfliers=True,euclidian_distance=target_variable)

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            boxplot(data,i,showfliers=False,euclidian_distance=target_variable)

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            histplot(data,i,euclidian_distance=None)

### EXPLORE CORROLEATION BETWEEN SAME EUCLIDIAN DISTANCE AND DTW BETWEEEN SAME POSITION

In [ ]:
pos_indices

In [ ]:

CorrelationMatrixPerDistance("smoothed VOC",X_whole,indices)

In [ ]:
DTWscoreSamePositionDifferentSensors("smoothed VOC",X_whole,pos_indices)

## VOC SMOOTHED normalize

### GRAB THE DATA

In [ ]:
column_to_take = "VOC smoothed"
number_of_distances_to_grab = 0
grab_closest = False
type_of_scaling="robust"
X_whole,Y_whole,indices,pos_indices =grab_X_whole_Y_whole(column_to_take,number_of_distances_to_grab,grab_closest,type_of_scaling="None",
                         userInputData_Original=userInputData_Original,timeSeriesData_BIG_Original=timeSeriesData_BIG_Original)



In [ ]:
X_whole

In [ ]:
Y_whole

In [ ]:
indices

In [ ]:
pos_indices

### DATA DISTRIBUTION PLOT FOR ALL DATA

In [ ]:
for i,data in enumerate(X_whole):

    boxplot(data,i,showfliers=True,euclidian_distance=None)

In [ ]:
for i,data in enumerate(X_whole):

    boxplot(data,i,showfliers=False,euclidian_distance=None)

In [ ]:
for sensor_id, data in enumerate(X_whole):
     histplot(data,sensor_id,euclidian_distance=None)


### DATA DISTRIBUTION PLOT FOR EVERY DIFFERENT DISTANCE

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            boxplot(data,i,showfliers=True,euclidian_distance=target_variable)

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            boxplot(data,i,showfliers=False,euclidian_distance=target_variable)

In [ ]:
print("use smoothed VOC")
for i,groups_based_on_distance in indices.items():
    
    for target_variable,group_indexes in groups_based_on_distance.items():
            idx = np.array(group_indexes)
            data = X_whole[i,idx]
            histplot(data,i,euclidian_distance=None)

### EXPLORE CORROLEATION BETWEEN SAME EUCLIDIAN DISTANCE AND DTW BETWEEEN SAME POSITION

In [ ]:
pos_indices

In [ ]:

CorrelationMatrixPerDistance("smoothed VOC",X_whole,indices)

In [ ]:
DTWscoreSamePositionDifferentSensors("smoothed VOC",X_whole,pos_indices)

In [ ]:
all_dfs[0]

In [ ]:
all_dfs[1]